# 🤖 Multi-AI Agent & Tool Calling System with LangGraph

Yeh notebook LangGraph ke through stateful agents, tool integration (`ToolNode`), conditional routing (`tools_condition`), aur agent execution workflows ko implement karta hai.

### Environment Setup & Model Initialization
- **Purpose**: `.env` file se API keys load karna aur LLM model ko initialize karna.
- **Model Used**: `openai/gpt-oss-20b` via Groq backend.
- **Explanation**: Yeh model tool-calling capabilities support karta hai jisse agent reasoning ke basis par tools execute kar sake.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables from .env
load_dotenv()

# Initialize Chat Model dynamically with Groq backend
model = init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')
print("Model initialized successfully!")

### Tool Calling & Graph Imports
- **Purpose**: LangGraph workflow aur tool execution ke liye core classes aur functions import karna.
- **Key Modules**:
  - `StateGraph`, `START`, `END`: Graph workflow aur state machine control.
  - `add_messages`: Message history ko overwrite karne ke bajaye append karne wala reducer.
  - `ToolNode`, `tools_condition`: Prebuilt tool execution node aur conditional routing helper.
  - `@tool`: Python functions ko LLM-callable tools mein convert karta hai.
  - `BaseMessage`, `HumanMessage`: LangChain ke standard message formats.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.tools import Tool, tool
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage

### `state` Schema Definition (TypedDict with Reducer)
- **Purpose**: Graph execution ke dauran shared state aur conversation history hold karna.
- **Fields**:
  - `messages`: `Annotated[list[BaseMessage], add_messages]` - Har naya message state mein existing list ke end mein append hota hai.
- **Explanation**: `add_messages` reducer ensure karta hai ki purani messages retain rahein aur context loss na ho.

In [ ]:
class state(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### `@tool` Definition & `ToolNode` Initialization
- **Function Name**: `add(a: float, b: float)`
- **Purpose**: Do numbers ko add karke unka mathematical sum return karna.
- **Parameters**:
  - `a` (`float`): First operand
  - `b` (`float`): Second operand
- **Return Value**: `float` (Sum of `a` and `b`)
- **`ToolNode`**: Prebuilt node jo LLM ke generated `tool_calls` ko receive karke actual Python function execute karta hai.

In [ ]:
@tool
def add(a: float, b: float):
    """add two numbers"""
    return a + b

tool_node = ToolNode([add])

### Tool Binding (`bind_tools`) & LLM Node
- **Function Name**: `call_llm_model(state)`
- **Purpose**: Model ko tools ke schema ke sath bind karna aur current message history pass karke response ya tool call generate karna.
- **Parameters**:
  - `state` (`state`): Current graph state containing conversation messages.
- **Return Value**: `dict` containing updated `messages` list with LLM's response.
- **Explanation**: `model.bind_tools([add])` model ko aware karta hai ki `add` tool available hai invocation ke liye.

In [ ]:
model_with_tool = model.bind_tools([add])

# Node definition for invoking model with tool binding
def call_llm_model(state: state):
    return {'messages': [model_with_tool.invoke(state['messages'])]}

### `StateGraph` Construction & Routing
- **Purpose**: State schema ke basis par workflow structure initialize karna, nodes add karna, aur conditional routing setup karna.
- **Components**:
  - Nodes: `'call_llm_model'`, `'tools'`
  - Edges: `START -> 'call_llm_model'`
  - Conditional Edge: `tools_condition` route karta hai ki agar LLM ne tool call kiya hai toh `'tools'` par jaye, warna `END` par jaye.
  - Return Edge: `'tools' -> 'call_llm_model'` (Tool execution ke baad output model ke paas wapas jata hai final response ke liye).

In [ ]:
builder = StateGraph(state)
builder.add_node('call_llm_model', call_llm_model)
builder.add_node('tools', tool_node)

builder.add_edge(START, 'call_llm_model')
builder.add_conditional_edges('call_llm_model', tools_condition)
builder.add_edge('tools', 'call_llm_model')

graph = builder.compile()

### Agent Invocation & Message Flow Test
- **Purpose**: Graph ko run karke verify karna ki model user query se tool call detect karta hai, tool execute hota hai, aur final conversational answer return hota hai.
- **Parameters**:
  - `inputs`: `dict` jisme `messages` list hoti hai (e.g., `HumanMessage(content="What is 45.5 + 23.8?")`).
- **Output**: Complete conversation trace with Tool calls and Tool responses.

In [ ]:
# Test invocation with a math query requiring the add tool
inputs = {"messages": [HumanMessage(content="What is 45.5 + 23.8?")]}
output = graph.invoke(inputs)

for m in output["messages"]:
    m.pretty_print()